# Condition-specific genes analysis using Multicellular Transformer

This tutorial show how one can analyse condition-specific genes using Multicellular Transformer.
Here, we measure condition-specificity using the gene neighborhood in a sample.
If a gene has a high neighborhood overlap in the samples coming from the same condition, where conditions may include healthy vs disease, or for example different tissue,
compared to the neighborhood overlap across conditions, then the gene is condition-specific.

By leveraging neighborhood, we can account for the network effects and measure a change in the gene across conditions not only using its expression, but also the expression of its neighborhood.

In this tutorial we show how one can:
* Analyse condition-specific genes in 3 different datasets using precomputed condition-specificity.
* How one can compute condition-specificity given neighborhood of genes across samples.


Before starting, make sure that you have the repo installed.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

### Condition-specific genes analysis using precomputed condition-specificity

We share the condition-specificity analysis done on 3 different datasets:
* Immune across organs ([Mapping the developing human immune system across organs](https://cellxgene.cziscience.com/e/fd072bc3-2dfb-46f8-b4e3-467cb3223182.cxg/))
* Systemic lupus erythematosus ([Single-cell RNA-seq reveals the cell-type-specific molecular and genetic associations to lupus](https://cellxgene.cziscience.com/collections/436154da-bcf1-4130-9c8b-120ff9a888f2))
* COVID-19 ([Single-cell multi-omics analysis of the immune response in COVID-19](https://cellxgene.cziscience.com/e/c7775e88-49bf-4ba2-a03b-93f00447c958.cxg/))

#### Immune across organs ([Mapping the developing human immune system across organs](https://cellxgene.cziscience.com/e/fd072bc3-2dfb-46f8-b4e3-467cb3223182.cxg/))

In [ ]:
# read the data
immune_across_organs_df = pd.read_parquet("data/immune_across_organs_condition_specific_genes.parquet")
# the condition specificity scores is calculated by dividing overlap_within_condition by overlap_across_conditions
immune_across_organs_df.head()

Plot overlap within vs overlap across conditions

In [ ]:
immune_across_organs_df['1 - overlap_across_conditions'] = 1 - immune_across_organs_df['overlap_across_conditions']

sns.jointplot(
    immune_across_organs_df,
    x='overlap_within_condition',
    y='1 - overlap_across_conditions',
    s=25,
    height=8,
    color="#003399",
    edgecolor=None,
    alpha=0.6,
)

plt.xlabel("Jaccard similarity between gene neighborhoods in the same condition", fontsize=12)
plt.ylabel("1 - Jaccard similarity between gene neighborhoods across conditions", fontsize=12)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.show()

#### Plot gene expression across tissues of highly tissue-specific genes

We will use the dataset downloaded from CELL x GENE ([Mapping the developing human immune system across organs](https://cellxgene.cziscience.com/e/fd072bc3-2dfb-46f8-b4e3-467cb3223182.cxg/)) and the highly tissue-specific genes read above.

In [ ]:
# read the immune across organs data, replace the directory in which the H5AD object is
adata = sc.read_h5ad(
    "/Users/wiatrakm/Downloads/7af3a87a-c148-4988-a7cd-f33666ffd883.h5ad", backed="r"
)
# set var names to 'GeneName' column instead of ENSEMBL ID
adata.var_names = adata.var['GeneName'].tolist()
# make the var names unique for plotting
adata.var_names_make_unique()

In [ ]:
# select top k tissue specific genes
k = 20
tissue_specific_genes = immune_across_organs_df['gene_name'].tolist()[:k]
# plot a dot plot
sc.pl.dotplot(adata, tissue_specific_genes, groupby='tissue', dendrogram=False, use_raw=False)

#### Systemic lupus erythematosus ([Single-cell RNA-seq reveals the cell-type-specific molecular and genetic associations to lupus](https://cellxgene.cziscience.com/collections/436154da-bcf1-4130-9c8b-120ff9a888f2))

In [ ]:
# read the data
lupus_df = pd.read_parquet("data/lupus_disease_condition_specific_genes.parquet")
lupus_df.head()

#### COVID-19 ([Single-cell multi-omics analysis of the immune response in COVID-19](https://cellxgene.cziscience.com/e/c7775e88-49bf-4ba2-a03b-93f00447c958.cxg/))

In [ ]:
# read the data
covid_df = pd.read_parquet("data/covid_stephenson_disease_condition_specific_genes.parquet")
covid_df.head()

## Compute condition-specificity given neighborhood of genes across samples

Below is an example of how given top k neighbors for genes across samples one can compute condition specificity.

#### Read data sample we will use for this experiment

This sample was computed by running the `utils/sample_embeddings/run_collect_sample_neighbors.py` script present in the repository on the immune across organs dataset ([Mapping the developing human immune system across organs](https://cellxgene.cziscience.com/e/fd072bc3-2dfb-46f8-b4e3-467cb3223182.cxg/)). We then extracted a subset of N genes from the resulting file for this dataset.

In [ ]:
neighbors_df = pd.read_parquet("data/immune_across_organs_neighbors_sample.parquet")
neighbors_df.head()

Calculate condition-specificity on a sample of genes

In [ ]:
from cell_cell_interactions.tl.condition_specific_genes import calculate_condition_specificity

# calculate condition specificity
condition_specificity_df = calculate_condition_specificity(
    df=neighbors_df,
    condition="tissue",
    min_n_conditions=2,
    min_n_samples=4,
    k_neighbors_within_condition=20,
    k_neighbors_across_conditions=40,
)
condition_specificity_df.head()

#### Plot the similarity across samples

In [ ]:
from cell_cell_interactions.pl.condition_specific_genes import plot_similarity_across_conditions_heatmap


# subset to a gene of interest
gene_df = neighbors_df[neighbors_df['gene_name'] == 'OIT3']
# plot jaccard similarity across conditions, the color bar represents jaccard similarity
# the x and y axis represent samples and each value in the matrix the overlap between them
plot_similarity_across_conditions_heatmap(
    gene_df=gene_df,  # dataframe with neighbors
    k=20,  # number of neighbors to consider
    min_items=3,  # minimum number of samples per gene in a condition
    condition='tissue',  # the column name for the condition
    labels_fontsize=16,  # font size for labels
)